# Selenium + Gemini：丝芙兰 AI 美妆顾问（社区贡献）

## 练习目标（理念）

在 Day 1「给 URL → 拿网页文本 → 让模型摘要」的骨架上，把抓取换成 **Selenium**（真实浏览器渲染 JS），再让 **Gemini**（OpenAI 兼容接口）扮演美妆顾问：

- **输入**：丝芙兰页面内容 + 顾客预算/需求
- **输出**：Markdown 格式的产品建议
- **为什么用 Selenium**：很多站点（含电商）靠 JS 渲染，简单 `requests` 拿不到正文

## 和本课 Day 1 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| Chat Completions | `openai.chat.completions.create` |
| `messages`（system / user） | 顾问角色 + 顾客需求 + 网页片段 |
| 网页抓取 | Selenium 取 `page_source`，BeautifulSoup 去噪 |
| 兼容 API | Gemini 的 OpenAI 兼容 `base_url` |

## 怎么跑

1. 安装 Chrome / ChromeDriver（本笔记本用 `webdriver_manager` 自动拉驱动）
2. `.env` 里配置 `GEMINI_API_KEY`
3. 从上到下运行；社区贡献核心在后半段「Selenium 打开 Sephora → 顾问推荐」
4. 前半段仍是 Day 1 预热（连通 API、消息结构、简单摘要）

更多课程 FAQ / 安装说明见仓库 `README.md`、`guides/`、`setup/troubleshooting.ipynb`。


### 环境准备（Cursor / VS Code）

1. 菜单 **View → Extensions**，安装 **Python**（ms-python）与 **Jupyter**（ms-toolsai）
2. 右上角 **Select Kernel** → **Python Environments…** → 选带 `.venv` 的解释器（推荐 3.12）
3. 每个笔记本都要单独选一次内核

若选核失败，先查 `setup/troubleshooting.ipynb`。


In [2]:
# ========== 导入 + 创建 Gemini 客户端（OpenAI 兼容） ==========

# 标准库 os：读环境变量里的 API Key
import os
# load_dotenv：把 .env 密钥注入进程，避免把密钥写进笔记本
from dotenv import load_dotenv
# 课程自带抓取工具：简单 HTTP 取站（对强 JS 站可能不够）
from scraper import fetch_website_contents
# IPython 展示：把模型返回的 Markdown 渲染出来
from IPython.display import Markdown, display
# OpenAI 客户端类：也可指向 Gemini 兼容端点
from openai import OpenAI

# override=True：.env 里的值覆盖已有环境变量
load_dotenv(override=True)

# 变量名仍叫 openai：与课程 Day1 习惯一致；实际走 Gemini
openai = OpenAI(
    api_key=os.getenv("GEMINI_API_KEY"),
    # Google Generative Language 的 OpenAI 兼容 Base URL（字符串勿改）
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)


ModuleNotFoundError: No module named 'openai'

# 连接到 Gemini（OpenAI 兼容）或改用 Ollama

下一格会再次 `load_dotenv` 并**检查** `GEMINI_API_KEY` 是否存在、是否误带空格。

- 想用免费本地模型：看 README「付费 API 的免费替代方案」，或参考 `day1_with_ollama.ipynb`
- 若报 `NameError`：多半是没从上到下跑全格
- API 费用：Day1 调用量通常很低；也可改走 Ollama


In [13]:
# ========== 校验 GEMINI_API_KEY：缺了/有空格就给出可读提示 ==========

# 再加载一次 .env（与上格一致；重复调用无害）
load_dotenv(override=True)
# 从环境读取密钥字符串
api_key = os.getenv("GEMINI_API_KEY")

# 分支提示保留英文：这是运行时给用户看的诊断文案，且与原逻辑一致
if not api_key:
    print("No Gemini API key was found. Please check your .env file.")
elif api_key.strip() != api_key:
    print("The Gemini API key has leading or trailing spaces. Please remove them.")
else:
    print("Gemini API key found and looks good!")


Gemini API key found and looks good!


# 先热身：对 Frontier / Gemini 发一条最小消息


In [14]:
# ========== 构造 messages：最少只要一条 user ==========

# 发给模型的内容保持英文（影响回答的字符串不翻译）
message = "Hello, Gemini! This is my first ever message to you! Hi!"

# OpenAI 风格：列表里每个元素是 {role, content}
messages = [{"role": "user", "content": message}]

# 笔记本里最后一行表达式会显示该对象，便于目检结构
messages


[{'role': 'user',
  'content': 'Hello, Gemini! This is my first ever message to you! Hi!'}]

In [15]:
# ========== 第一次真实 API 调用：非流式 Chat Completions ==========

# model 字符串必须是端点认识的模型 id
response = openai.chat.completions.create(
    model="gemini-2.5-flash",
    messages=messages
)

# choices[0].message.content：第一条候选回复的正文
response.choices[0].message.content


"Hello there! It's fantastic to meet you!\n\nWelcome! I'm really glad you reached out. How can I help you today, or what's on your mind?"

## 开始第一个小项目：抓网页 → 再摘要


In [16]:
# ========== 试用课程 scraper：HTTP 拉取个人站正文 ==========

# fetch_website_contents：封装了请求与粗清洗；对 JS 重站可能失败
ed = fetch_website_contents("https://edwarddonner.com")
# 打印一小段，确认抓取通路正常
print(ed)


Home - Edward Donner

Skip to content
Avatar
Curriculum
Proficiency
C4
Outsmart
An arena that pits LLMs against each other in a battle of diplomacy and deviousness
About
Posts
Well, hi there.
I’m Ed. I like writing code and experimenting with LLMs, and hopefully you’re here because you do too. I also enjoy amateur electronic music production (
very
amateur) and losing myself in
Hacker News
, nodding my head sagely to things I only half understand.
I’m the co-founder and CTO of AI startup
Nebula.io
. I was previously founder and CEO of AI startup untapt,
acquired in 2021
, and a Managing Director at JPMorgan.
I will happily drone on for hours about LLMs to anyone in my vicinity. My friends got fed up with my impromptu lectures, and convinced me to make some Udemy courses. To my total joy (and shock) they’ve become best-selling, top-rated courses, with 600,000 enrolled across 194 countries. The
full curriculum is here
. If you’re visiting from one of my courses – I’m super grateful!
For 

## 提示词类型（Prompt Types）

Frontier 模型习惯两种指令角色：

- **System prompt**：规定任务、语气、输出格式（你是谁、怎么答）
- **User prompt**：本轮具体输入（网页正文、用户问题等）

后面会把两者拼进 `messages` 列表再调用 API。


In [33]:
# ========== 定义 system_prompt：任务 + 语气（发给模型，保留英文） ==========

# 可自行改最后一句，例如要求 Spanish / 更礼貌；但不要在注释里「翻译掉」可执行字符串
system_prompt = """
You are a rude assistant that analyzes the contents of a website,
and provides a short, snarky, humorous summary, ignoring text that might be navigation related.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""


In [34]:
# ========== 定义 user_prompt 前缀：后面会拼接网页正文 ==========

user_prompt_prefix = """
Here are the contents of a website.
Provide a short summary of this website.
If it includes news or announcements, then summarize these too.

"""


## Messages 结构

OpenAI（以及许多兼容 API）期望：

```python
[
    {"role": "system", "content": "system message goes here"},
    {"role": "user", "content": "user message goes here"}
]
```

下面两格先用一个「莎士比亚口吻算 2+2」的玩具例子熟悉调用形状。


In [24]:
# ========== 玩具调用：system 定口吻，user 提问 ==========

messages = [
    {"role": "system", "content": "You are an assistance who speaks like Shakespeare"},
    {"role": "user", "content": "What is 2 + 2?"}
]

# 同一客户端、同一模型 id；只换 messages
response = openai.chat.completions.create(model="gemini-2.5-flash", messages=messages)
response.choices[0].message.content


"Hark! When two is added unto two, the sum that doth appear is **four**! Forsooth, 'tis a truth as plain as day."

## 用函数拼出「摘要网站」所需的 messages


In [25]:
# ========== messages_for：把 system +（前缀+网页正文）打成 API 列表 ==========

def messages_for(website):
    # website 这里是字符串正文，不是 URL
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + website}
    ]


In [26]:
# ========== 目检：对刚才抓到的 ed 正文生成 messages ==========

messages_for(ed)


[{'role': 'system',
  'content': '\nYou are a snarky assistant that analyzes the contents of a website,\nand provides a short, snarky, humorous summary, ignoring text that might be navigation related.\nRespond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.\n'},
 {'role': 'user',
  'content': '\nHere are the contents of a website.\nProvide a short summary of this website.\nIf it includes news or announcements, then summarize these too.\n\nHome - Edward Donner\n\nSkip to content\nAvatar\nCurriculum\nProficiency\nC4\nOutsmart\nAn arena that pits LLMs against each other in a battle of diplomacy and deviousness\nAbout\nPosts\nWell, hi there.\nI’m Ed. I like writing code and experimenting with LLMs, and hopefully you’re here because you do too. I also enjoy amateur electronic music production (\nvery\namateur) and losing myself in\nHacker News\n, nodding my head sagely to things I only half understand.\nI’m the co-founder and CTO of AI startup\nNebula

## 拼起来：抓取 → messages → Chat Completions


In [27]:
# ========== summarize(url)：端到端摘要一条 URL ==========

def summarize(url):
    # 1) 抓网页文本
    website = fetch_website_contents(url)
    # 2) 非流式调用；messages_for 内部用到上面的 system/user 模板
    response = openai.chat.completions.create(
        model = "gemini-2.5-flash",
        messages = messages_for(website)
    )
    # 3) 只返回助手文本
    return response.choices[0].message.content


In [28]:
# ========== 试跑：摘要 edwarddonner.com ==========

summarize("https://edwarddonner.com")


'This website belongs to Edward Donner, a man who loves LLMs so much his friends made him turn his incessant droning into best-selling Udemy courses. He\'s an AI bigwig, a startup success story, and apparently, a "very amateur" electronic musician who forces large language models to battle each other for his amusement. He even has a digital avatar, presumably to lecture you when the real Ed is busy losing himself in Hacker News.\n\n**Future Announcements (yes, future):**\nLooks like Ed\'s already planning content for 2025 and 2026, so get ready for resources on becoming an "Agentic Engineer," building AI with n8n, and deploying AI to production. The biggest question, though, isn\'t about the tech, but "Which order to take the AI courses?" – a dilemma from May 28, 2025, that we\'re all clearly stressed about now.'

In [29]:
# ========== display_summary：摘要后再用 Markdown 渲染 ==========

def display_summary(url):
    summary = summarize(url)
    # display(Markdown(...))：在 Jupyter 里渲染标题/列表
    display(Markdown(summary))


In [37]:
# ========== 渲染版摘要：同一站点 ==========

display_summary("https://edwarddonner.com")


This is Edward Donner's digital soapbox, where he'll happily drone on about his "best-selling" Udemy courses on LLMs (600,000 students, whoopie!), his AI startup adventures, and his "very amateur" electronic music. He also enjoys nodding sagely at Hacker News, probably pretending he understands it all. If you can't get enough of him, you can even chat with his digital avatar – talk about self-love.

As for news, this guy's apparently a time traveler. He's already announced future "resources" for his AI Coder, Builder, and Engineering tracks, helpfully dated from 2025 and 2026. Because waiting for content is for amateurs.

# 再试更多网站

简单 HTTP 抓取**只能**对付「服务端直接吐 HTML」的站点。

- **React 等前端渲染站**：正文可能不在首包 HTML 里 → 需要 Selenium / Playwright（见本笔记本后半社区贡献）
- **CloudFront 等防护**：可能 403
- 许多新闻/营销站仍可用 `display_summary` 直接试


In [35]:
# ========== 试 CNN ==========

display_summary("https://cnn.com")


This is CNN, a groundbreaking digital experience where the most pressing "breaking news" appears to be their consistently broken video player and ads that inflict more pain than information. It's a vast wasteland of categories for literally everything under the sun, ensuring you'll find exactly what you didn't know you needed, while they beg for feedback on why their site is perpetually failing.

In [36]:
# ========== 试 Anthropic 官网 ==========

display_summary("https://anthropic.com")


Oh look, another AI company, Anthropic, that "puts safety at the frontier" while churning out a whole stable of AI products like Claude (and its many tedious variations), Mythos, and Fable. They're crowing about "Announcing Fable 5," which apparently is the "next generation of intelligence." Even better, they've already had to "Redeploy Fable 5" because some export controls were lifted, so now everyone globally can get their hands on it starting July 1, 2026. Riveting stuff.

## 商业应用与练习提示

你已经走通了：**Cloud API + 网页文本 → 生成摘要**。摘要是经典 GenAI 用例（新闻、财报、简历/求职信……）。

**动手**：下面单元格请改成你自己的业务小例子。原课常见作业是「根据邮件正文建议短主题行」；本笔记本示例改成了「租房网站筛选」。


In [39]:
# ========== 业务小例子：租房筛选（prompt 保留英文，改译会改行为） ==========

# Step 1：写 system / user 提示词
system_prompt = """
You are a polite, friendly, and helpful real estate assistant.

Your task is to analyze the contents of a real estate website and identify apartments that match the user's requirements.

Only recommend listings that satisfy ALL of the following:
- Available for rent
- Suitable for students whenever possible
- Maximum monthly rent of €600
- Shared apartment preferred (studio is acceptable if no one-bedroom apartment is available)

For each matching listing, provide:
- Monthly rent
- Location
- Property type
- Number of bedrooms
- Furnished or unfurnished (if available)
- A short summary of the property

If no suitable apartments are found, clearly explain that there are no matching listings and briefly mention the closest alternatives.
"""

user_prompt = """
Please analyze the following real estate website and summarize all rental listings that meet these requirements:

- Budget: up to €600 per month
- One-bedroom apartment preferred (studio is acceptable)
- Suitable for students whenever possible

Website:
https://www.lacartedescolocs.fr/
"""

# Step 2：组装 messages
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt}
]

# Step 3：调用 Gemini（OpenAI 兼容）
response = openai.chat.completions.create(
    model="gemini-2.5-flash",
    messages=messages
)

# Step 4：打印纯文本结果
print(response.choices[0].message.content)


Hello there! I'd be happy to help you find suitable rental listings.

Please note that as an AI, I cannot browse live, real-time websites directly. The link you provided, "https://www.lacartedescolocs.fr/", is a website dedicated to **colocations** (flatshares). This means it primarily lists individual rooms for rent within shared apartments, which are often a very popular and budget-friendly option for students.

While your preference was for a one-bedroom apartment or a studio, given the nature of this particular website, the most common listings you would find within your budget would be rooms in shared apartments. I will provide examples that reflect what you would *likely* find on such a site, keeping your budget and student suitability in mind.

Based on typical listings on "La Carte des Colocs" that meet your criteria:

---

### Matching Rental Listings (Illustrative Examples)

Here are some examples of listings you might find that fit your requirements for a budget of up to €60

## 加分练习：强 JS 站点与 Selenium

若 `display_summary("https://openai.com")` 失败，常见原因是页面靠 JavaScript 渲染。可用 **Selenium / Playwright** 在真实浏览器里打开页面再取 DOM。

社区贡献文件夹里有同学提交的 Selenium 版本；本笔记本后半就是「Selenium + Sephora + 美妆顾问」示例。


# 社区贡献：Selenium + Gemini 的 AI 美妆顾问

扩展 Day 1 摘要器：用 **Selenium** 打开支持 JS 的电商页，再用 **BeautifulSoup** 清洗，最后让 **Gemini** 基于页面文本做推荐。

工作流：

1. Selenium 加载页面并取 `page_source`
2. BeautifulSoup 去掉 `script` / `style` 等，抽出纯文本
3. 构造美妆顾问 system/user messages
4. `display(Markdown(...))` 展示建议


In [2]:
# ========== 社区核心：Selenium 打开 Sephora → 清洗 HTML → 展示文本 ==========

# os：读 GEMINI_API_KEY
import os
# time.sleep：粗暴等待页面 JS 渲染（生产可换显式等待）
import time

# 加载 .env
from dotenv import load_dotenv

# Selenium：驱动真实 Chrome
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
# webdriver_manager：自动下载匹配的 ChromeDriver
from webdriver_manager.chrome import ChromeDriverManager

# BeautifulSoup：解析 HTML、删标签、抽文本
from bs4 import BeautifulSoup

# Jupyter 里渲染 Markdown
from IPython.display import Markdown, display

# OpenAI 兼容客户端（此处指向 Gemini）
from openai import OpenAI

# 读入 API Key
load_dotenv(override=True)

# 新建客户端：密钥 + Gemini OpenAI 兼容 Base URL
client = OpenAI(
    api_key=os.getenv("GEMINI_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

# 目标站点：丝芙兰首页（影响抓取的 URL 保持原样）
url = "https://www.sephora.com"

# 启动 Chrome：Service 包装驱动路径
driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install())
)

# 导航到 URL（会执行页面 JS）
driver.get(url)

# 等待 5 秒让首屏内容出来（简陋但课程演示够用）
time.sleep(5)

# 取渲染后的完整 HTML 字符串
html = driver.page_source

# 用完即关，避免残留浏览器进程
driver.quit()

# 解析 HTML 为可查询的树
soup = BeautifulSoup(html, "html.parser")

# 删掉脚本/样式/noscript，减少噪声
for tag in soup(["script", "style", "noscript"]):
    tag.decompose()

# 抽出可见文本：换行分隔并 strip
text = soup.get_text(separator="\n", strip=True)

# 只展示前 3000 字，避免笔记本被整页淹没
display(Markdown(f"""
# 💄您的美容顾问

# # 🇨🇳丝芙兰主页

{text[:3000]}
"""))



# 💄 Your Beauty Advisor

## 🌸 Sephora Homepage

Makeup, Skincare, Fragrance, Hair & Beauty Products | Sephora
Buy 2 Minis, Get Both 50% Off
.
∆
Mix and match up to 48 brands.
∆
Ends 7/15.
∆
Terms apply.
SHOP NOW
▸
Search
Shop Store & Delivery
Choose your store & location
Services & Events
Sign In
or
Join
for FREE Shipping 🚚
New
Minis Sale
Makeup
Skincare
Fragrance
Hair
Bath & Body
Mini Size
Brands
Gifts & Value Sets
Gift Cards
Sale & Offers
Added for Get It Shipped
View Basket
Checkout
Home
Shop
Offers
Gallery
My Store
Sephora Homepage
Good afternoon, Beautiful. 👋
Join Beauty Insider
to earn points with every purchase.
Shop
My Store
New
Minis Sale
Makeup
Skincare
Fragrance
Hair
Bath & Body
Mini Size
Brands
Gifts & Value Sets
Gift Cards
Chosen For You
Quicklook
rhode
Pocket Bronze Long-Wearing Cream Bronzer
$25.00
268
New
Quicklook
Yves Saint Laurent
Libre Eau De Parfum with Orange Blossom & Lavender
$38.00 - $225.00
4.5K
Quicklook
CHANEL
GABRIELLE CHANEL ESSENCE Eau de Parfum
$121.00 - $195.00
136
Quicklook
Marc Jacobs Beauty
Perfect Eau de Parfum with Daffodil & Musk
$37.00 - $165.00
1.3K
Quicklook
LANEIGE
Lip Sleeping Mask – Intense Hydration Lip Treatment with Vitamin C
$24.00 - $25.00
22.3K
New
Limited Edition
Quicklook
Valentino
Donna Born in Roma Eau de Parfum with Bourbon Vanilla & Jasmine
$37.00 - $180.00
2.9K
Quicklook
Dr. Dennis Gross Skincare
Alpha Beta® Extra Strength Daily Peel Pads
$20.00 - $155.00
($102.00 value)
7.8K
Quicklook
Kiehl's Since 1851
Ultra Facial Cream with SPF 30 Sunscreen
$39.00 - $72.00
405
Quicklook
Farmacy
Green Clean Makeup Removing Cleansing Balm
$13.50
$18.00 - $68.00
959
Quicklook
Glow Recipe
Watermelon Glow PHA + BHA Pore-Tight Toner
$16.00 - $36.00
8.1K
New
Limited Edition
Beauty Offers (17)
View all
Save Big on Minis
Buy 2, get both 50% off.
Δ
Cherry on top? You can mix and match up to 48 brands.
Δ
In store & online •
Ends 7/15/26
Δ
Terms apply.
Shop Minis
See details
7 Days Left
Get 4X Points
††1
on all Benefit Cosmetics.
Beauty Insider members only.
In store & online •
Ends 7/15/26
††1
Exclusions/terms apply. May be combined with other promotional offers.
Apply
See details
7 Days Left
Get 4X Points
††
on all Dolce&Gabbana.
Beauty Insider members only.
In store & online •
Ends 7/15/26
††
Exclusions/terms apply. May be combined with other promotional offers.
Apply
See details
7 Days Left
Get 5X Points
††2
on all Sephora Collection—only on the app.
Beauty Insider members only.
App only •
Ends 7/15/26
††2
Exclusions/terms apply. May be combined with other promotional offers.
Download App
See details
7 Days Left
Try Beauty of Joseon
Get a free trial size of the fan-fave Day Dew Sunscreen Lightweight SPF 50.
Free with $30+ purchase.*
Beauty Insider members only.
Online only
*Exclusions/terms apply.
Apply
Free Sol de Janeiro Mini
Spritz on Cheirosa 62 Perfume Mist for notes like pistachio and salted caramel.
Free with $30 purchase.*
Beauty Insider members only.
App only
*Exclusions/terms apply.
Download App
Get 15% off¶ Laura Mercier
Subscribe and save on sel


In [3]:
# ========== 美妆顾问：用页面文本 + 顾客预算调用 Gemini ==========

# system：顾问人设与约束（发给模型，保留英文）
system_prompt = """
You are a friendly and professional AI Beauty Advisor for Sephora.

Help customers choose beauty products based on the information they provide.

If the customer provides their skin type, use it when recommending products.

If the customer does not provide their skin type, mention that your recommendations are general and ask them to share their skin type for more personalized suggestions.

Recommend only products available on the Sephora website.

Format your response in clear Markdown.
"""

# user：顾客需求 + 截断后的网页文本（控制 token）
user_prompt = f"""
Customer Request:

My budget is $50.

Show me some moisturizers available on Sephora within my budget.

Website Information:

{text[:4000]}
"""

# 拼 messages
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt}
]
# 非流式生成推荐
response = client.chat.completions.create(
    model="gemini-2.5-flash",
    messages=messages
)
# 渲染助手 Markdown
display(Markdown(response.choices[0].message.content))


Hello there! I'd be happy to help you find a great moisturizer within your $50 budget at Sephora.

Based on your request, here's a popular option that fits your budget:

*   **Kiehl's Since 1851 Ultra Facial Cream with SPF 30 Sunscreen**
    *   **Price:** $39.00 (for the smaller size)
    *   **Why it's a good pick:** This cult-favorite moisturizer is known for providing 24-hour hydration and is formulated with SPF 30 to help protect your skin from sun exposure. It's often praised for its lightweight texture and suitability for a variety of skin types.

**Please note:** Since you haven't mentioned your skin type (e.g., oily, dry, combination, sensitive, normal), this is a general recommendation.

**For more personalized suggestions, please share your skin type!** Knowing this will allow me to recommend moisturizers that are best suited to your specific needs.

You might also want to explore the **Minis Sale** section on Sephora's website. Moisturizers are often available in smaller sizes, and with the current "Buy 2 Minis, Get Both 50% Off" offer, you could discover a new favorite or get a great deal within your budget!

Let me know if you have any other questions or if you'd like to share your skin type!

# 分享你的代码

若你改进了抓取/提示词，欢迎把变更放到 `community-contributions` 并发 PR。

提交前自检：

1. PR 尽量只含社区贡献相关改动
2. 笔记本输出不要过大
3. 总改动行数别爆（课程建议量级）
4. 不要提交 `.env`、冗长测试垃圾文件

PR 说明：https://edwarddonner.com/pr
